In [1]:
#Q1

import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", None)

roll_no = input("Enter your college roll number: ")
digits = [int(d) for d in roll_no]
last_two = digits[-2:]
categories = ["billing", "account", "general"]

print("Digits:", digits)
print("Last two digits:", last_two)
for d in last_two:
    print(f"digit {d} -> category[{d} % 3] = {categories[d % 3]}")

fixed_entries = [
 {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
 "keywords": "fee cost price charge", "category": "billing"},
 {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
 "keywords": "password reset login", "category": "account"},
 {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
 "keywords": "hours timing open time", "category": "general"},
 {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
 "keywords": "pay payment upi fee", "category": "billing"},
]

personalized_entries = [
    {"question": "what are your holiday hours",
     "answer": "We remain closed on all national holidays.",
     "keywords": "holiday hours closed",
     "category": categories[last_two[0] % 3]},
    {"question": "how do i contact customer support",
     "answer": "You can reach us via email at support@college.edu or call our helpdesk.",
     "keywords": "contact support helpdesk",
     "category": categories[last_two[1] % 3]},
]

faq_df = pd.DataFrame(fixed_entries + personalized_entries)
print()
print(faq_df)

Enter your college roll number: 1024170323


Digits: [1, 0, 2, 4, 1, 7, 0, 3, 2, 3]
Last two digits: [2, 3]
digit 2 -> category[2 % 3] = general
digit 3 -> category[3 % 3] = billing

                            question                                             answer                  keywords category
0             what is the annual fee                          The annual fee is Rs 500.     fee cost price charge  billing
1              how to reset password                   Go to Settings > Reset Password.      password reset login  account
2        what are your working hours                          We are open 9 AM to 5 PM.    hours timing open time  general
3              how can i pay the fee         You can pay via UPI, card, or net banking.       pay payment upi fee  billing
4        what are your holiday hours         We remain closed on all national holidays.      holiday hours closed  general
5  how do i contact customer support  You can reach us via email at support@college....  contact support helpdesk  billing


In [2]:
#Q2

def score_query(query, df):
    query_words = set(query.lower().split())
    results = []
    for idx, row in df.iterrows():
        kw_words = set(row["keywords"].lower().split())
        overlap = query_words & kw_words
        score = len(overlap)
        if score > 0:
            results.append((idx, row["question"], row["answer"], score))
    results.sort(key=lambda x: x[3], reverse=True)
    return results

query = "how do i pay my fee"
print("Query:", query)
matches = score_query(query, faq_df)
for m in matches:
    print(f"Score {m[3]}: Q: {m[1]} -> A: {m[2]}")

Query: how do i pay my fee
Score 2: Q: how can i pay the fee -> A: You can pay via UPI, card, or net banking.
Score 1: Q: what is the annual fee -> A: The annual fee is Rs 500.


In [3]:
#Q3

def same_category(category_name, df):
    return df[df["category"] == category_name]["question"].tolist()

target_category = personalized_entries[0]["category"]
print("Category used:", target_category)
print(same_category(target_category, faq_df))

Category used: general
['what are your working hours', 'what are your holiday hours']


In [4]:
#Q4

print(faq_df[["question", "keywords"]])

entry_index = 0
new_keyword = input(f"Enter a new keyword to add to entry {entry_index}: ")
faq_df.at[entry_index, "keywords"] = faq_df.at[entry_index, "keywords"] + " " + new_keyword

print("\nUpdated entry:")
print(faq_df.loc[entry_index])

csv_name = f"{roll_no}_faq_data.csv"
faq_df.to_csv(csv_name, index=False)
print(f"\nSaved to {csv_name}")

                            question                  keywords
0             what is the annual fee     fee cost price charge
1              how to reset password      password reset login
2        what are your working hours    hours timing open time
3              how can i pay the fee       pay payment upi fee
4        what are your holiday hours      holiday hours closed
5  how do i contact customer support  contact support helpdesk


Enter a new keyword to add to entry 0: refund



Updated entry:
question          what is the annual fee
answer         The annual fee is Rs 500.
keywords    fee cost price charge refund
category                         billing
Name: 0, dtype: str

Saved to 1024170323_faq_data.csv


In [5]:
#Q5

print(faq_df.groupby("category").size())

category
account    1
billing    3
general    2
dtype: int64


In [6]:
#Q6

def score_query_v2(query, df):
    query_words = set(query.lower().split())
    results = []
    for idx, row in df.iterrows():
        kw_words = set(row["keywords"].lower().split())
        overlap = query_words & kw_words
        score = len(overlap)
        if score > 0:
            results.append((idx, row["question"], row["answer"], score))
    if not results:
        print("No matching entries found.")
        return []

    max_score = max(r[3] for r in results)
    top_matches = [r for r in results if r[3] == max_score]

    if len(top_matches) > 1:
        print(f"Multiple entries tied with score {max_score}:")
        for m in top_matches:
            print(f" - {m[1]} -> {m[2]}")
    else:
        m = top_matches[0]
        print(f"Score {m[3]}: {m[1]} -> {m[2]}")

    return top_matches

tie_query = "fee"
print("Query (tie expected):", tie_query)
_ = score_query_v2(tie_query, faq_df)

print()
unique_query = "reset password"
print("Query (unique expected):", unique_query)
_ = score_query_v2(unique_query, faq_df)

Query (tie expected): fee
Multiple entries tied with score 1:
 - what is the annual fee -> The annual fee is Rs 500.
 - how can i pay the fee -> You can pay via UPI, card, or net banking.

Query (unique expected): reset password
Score 2: how to reset password -> Go to Settings > Reset Password.
